In [ ]:
# Environment setup
!pip install torch torchvision efficientnet-pytorch albumentations imbalanced-learn scikit-learn tqdm

import os, random, numpy as np, pandas as pd, time, json, cv2
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler
from albumentations import Compose, Resize, Normalize, HorizontalFlip, RandomBrightnessContrast, CLAHE, MixUp, ToTensorV2
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from efficientnet_pytorch import EfficientNet


In [ ]:
class BalancedChestXrayDataset(Dataset):
    def __init__(self, df, imgdir, transform=None, minorityboost=True):
        self.df = df.reset_index(drop=True)
        self.imgdir = imgdir
        self.transform = transform
        self.minorityboost = minorityboost
        self.labels = ['Atelectasis','Consolidation','Infiltration','Pneumothorax','Edema',
                       'Emphysema','Fibrosis','Effusion','Pneumonia','Pleural_Thickening',
                       'Cardiomegaly','Nodule','Mass','Hernia']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        imgpath = os.path.join(self.imgdir, row['Image Index'])
        img = np.array(Image.open(imgpath).convert('RGB'))

        # Enhanced preprocessing for minority class visibility
        minority_labels = ['Nodule', 'Mass', 'Hernia']
        if self.minorityboost and any(label in row['Finding Labels'] for label in minority_labels):
            img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
            img = cv2.equalizeHist(img)
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        # MixUp: randomly blend with other sample
        if random.random() < 0.3:
            j = random.randint(0, len(self.df) - 1)
            img2 = np.array(Image.open(os.path.join(self.imgdir, self.df.iloc[j]['Image Index'])).convert('RGB'))
            img = cv2.addWeighted(img, 0.7, img2, 0.3, 0)

        y = np.zeros(len(self.labels), dtype=np.float32)
        if row['Finding Labels'] != 'No Finding':
            for k, l in enumerate(self.labels):
                if l in row['Finding Labels']:
                    y[k] = 1
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(y)


In [ ]:
def create_balancedsampler(df, labels):
    classcounts = np.zeros(len(labels))
    for i, label in enumerate(labels):
        classcounts[i] = df['Finding Labels'].str.contains(label, na=False).sum()
    minority_boost_factor = 5  # boost rare class weights by 5x
    classweights = 1.0 / (classcounts + 1)
    rare_indices = [i for i, c in enumerate(classcounts) if c < len(df) * 0.05] # if class appears in <5%
    for idx in rare_indices:
        classweights[idx] *= minority_boost_factor
    classweights = classweights / classweights.sum() * len(labels)
    sampleweights = []
    for _, row in df.iterrows():
        labelspresent = [i for i, l in enumerate(labels) if l in row['Finding Labels']]
        if labelspresent:
            sampleweights.append(max([classweights[i] for i in labelspresent]))
        else:
            sampleweights.append(1.0)
    return WeightedRandomSampler(sampleweights, len(sampleweights), replacement=True)


In [ ]:
class AsymmetricFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma_pos=0, gamma_neg=2):
        super().__init__()
        self.alpha = alpha
        self.gamma_pos = gamma_pos
        self.gamma_neg = gamma_neg

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs)
        targets = targets.float()
        pt = inputs * targets + (1 - inputs) * (1 - targets)
        gamma = self.gamma_pos * targets + self.gamma_neg * (1 - targets)
        focal = (1 - pt) ** gamma
        BCE = F.binary_cross_entropy(inputs, targets, reduction='none')
        if self.alpha is not None:
            focal = focal * self.alpha
        loss = focal * BCE
        return loss.mean()


In [ ]:
class EffNetB3(nn.Module):
    def __init__(self, numclasses=14):
        super().__init__()
        self.net = EfficientNet.from_pretrained('efficientnet-b3')
        n = self.net._fc.in_features
        self.net._fc = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(n, numclasses))
    def forward(self, x):
        return self.net(x)


In [ ]:
tftrain = Compose([
    Resize(300, 300), HorizontalFlip(p=0.5), CLAHE(p=0.3),
    RandomBrightnessContrast(0.2, 0.2, p=0.5),
    Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])
tfval = Compose([
    Resize(300, 300),
    Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])


In [ ]:
def trainepoch(model, ldr, lossfn, opt, scaler):
    model.train()
    totloss, labs, preds = 0, [], []
    for x, y in tqdm(ldr, desc='Training', ncols=100):
        x, y = x.cuda(), y.cuda()
        opt.zero_grad(set_to_none=True)
        with autocast(device_type='cuda', dtype=torch.float16):
            out = model(x)
            loss = lossfn(out, y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        totloss += loss.item()
        labs.append(y.detach().cpu().numpy())
        preds.append(torch.sigmoid(out).detach().cpu().numpy())
    auc = roc_auc_score(np.vstack(labs), np.vstack(preds), average='macro')
    return totloss / len(ldr), auc

def valepoch(model, ldr, lossfn):
    model.eval()
    totloss, labs, preds = 0, [], []
    with torch.no_grad():
        for x, y in tqdm(ldr, desc='Validating', ncols=100):
            x, y = x.cuda(), y.cuda()
            with autocast(device_type='cuda', dtype=torch.float16):
                out = model(x)
                loss = lossfn(out, y)
            totloss += loss.item()
            labs.append(y.cpu().numpy())
            preds.append(torch.sigmoid(out).cpu().numpy())
    labs = np.vstack(labs)
    preds = np.vstack(preds)
    auc = roc_auc_score(labs, preds, average='macro')
    perclassauc = roc_auc_score(labs, preds, average=None)
    # Per-class threshold search
    thresholds = []
    for i in range(labs.shape[1]):
        fpr, tpr, th = roc_curve(labs[:,i], preds[:,i])
        best = th[np.argmax(tpr-fpr)]
        thresholds.append(best)
    return totloss / len(ldr), auc, perclassauc, thresholds


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
labels = ['Atelectasis','Consolidation','Infiltration','Pneumothorax','Edema',
          'Emphysema','Fibrosis','Effusion','Pneumonia','Pleural_Thickening',
          'Cardiomegaly','Nodule','Mass','Hernia']
trainds = BalancedChestXrayDataset(traindf, extract, tftrain, minorityboost=True)
valds = BalancedChestXrayDataset(valdf, extract, tfval, minorityboost=False)
sampler = create_balancedsampler(traindf, labels)

trainloader = DataLoader(trainds, batch_size=16, sampler=sampler, num_workers=4)
valloader = DataLoader(valds, batch_size=16, shuffle=False, num_workers=4)

# Strong class weights
classcounts = np.zeros(len(labels))
for i, l in enumerate(labels): classcounts[i] = traindf['Finding Labels'].str.contains(l, na=False).sum()
classweights = (len(traindf) / (len(labels) * (classcounts + 1))).astype(np.float32)
rare_indices = [i for i, c in enumerate(classcounts) if c < len(traindf)*0.05]
for idx in rare_indices: classweights[idx] *= 5
classweights = torch.FloatTensor(classweights).to(device)

model = EffNetB3().to(device)
lossfn = AsymmetricFocalLoss(alpha=classweights, gamma_pos=0, gamma_neg=2)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', patience=2, factor=0.5)
scaler = GradScaler(device='cuda')

bestauc = 0
for epoch in range(1, 21):  # train for more epochs with new backbone
    print(f'Epoch {epoch}')
    trloss, trauc = trainepoch(model, trainloader, lossfn, opt, scaler)
    valloss, valauc, perclassauc, thresholds = valepoch(model, valloader, lossfn)
    scheduler.step(valauc)
    print(f'Train Loss {trloss:.4f} AUC {trauc:.4f}')
    print(f'Val Loss {valloss:.4f} AUC {valauc:.4f}')
    print('Per-class AUC:', perclassauc)
    if valauc > bestauc:
        bestauc = valauc
        torch.save(model.state_dict(), 'best_efficientnetb3.pth')
        print('Best model saved!')
    # save thresholds for later use
    with open('best_thresholds.json', 'w') as f:
        json.dump(thresholds, f)
